# Build Website

This notebook combines the current analysis outputs, contextual statistics and
HTML template, then writes the local and GitHub Pages versions of the site.

Inputs:

- `output/geojson/combined.json`
- `output/sea_level_scenarios.json`
- `output/sea_level_trend.json`
- `data/reference/context_stats.json`
- `web/index_template.html`

## 1. Paths

In [1]:
import json
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "web" / "index_template.html").exists() and (candidate / "output").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")

PROJECT_ROOT = find_project_root()
COMBINED_JSON_PATH = PROJECT_ROOT / "output" / "geojson" / "combined.json"
STATS_JSON_PATH = PROJECT_ROOT / "output" / "sea_level_scenarios.json"
SEA_LEVEL_JSON_PATH = PROJECT_ROOT / "output" / "sea_level_trend.json"
CONTEXT_STATS_PATH = PROJECT_ROOT / "data" / "reference" / "context_stats.json"
TEMPLATE_PATH = PROJECT_ROOT / "web" / "index_template.html"
PREVIEW_PATH = PROJECT_ROOT / "web" / "index.html"
DEPLOY_PATH = PROJECT_ROOT / "docs" / "index.html"
VALIDATION_REPORT_PATH = PROJECT_ROOT / "output" / "data_validation_report.json"

for label, path in [
    ("Combined GeoJSON", COMBINED_JSON_PATH),
    ("Scenario statistics", STATS_JSON_PATH),
    ("Observed sea-level series", SEA_LEVEL_JSON_PATH),
    ("Context statistics", CONTEXT_STATS_PATH),
    ("HTML template", TEMPLATE_PATH),
]:
    print(f"{label}: {path} | exists={path.exists()}")

Combined GeoJSON: D:\2026 Pacific Dataviz Challenge\Git Hub\when-the-ocean-rises-github\output\geojson\combined.json | exists=True
Scenario statistics: D:\2026 Pacific Dataviz Challenge\Git Hub\when-the-ocean-rises-github\output\sea_level_scenarios.json | exists=True
Observed sea-level series: D:\2026 Pacific Dataviz Challenge\Git Hub\when-the-ocean-rises-github\output\sea_level_trend.json | exists=True
Context statistics: D:\2026 Pacific Dataviz Challenge\Git Hub\when-the-ocean-rises-github\data\reference\context_stats.json | exists=True
HTML template: D:\2026 Pacific Dataviz Challenge\Git Hub\when-the-ocean-rises-github\web\index_template.html | exists=True


## 2. Load inputs

In [2]:
with open(COMBINED_JSON_PATH, encoding="utf-8") as f:
    geo_data = f.read()

with open(STATS_JSON_PATH, encoding="utf-8") as f:
    scenarios_raw = json.load(f)

with open(SEA_LEVEL_JSON_PATH, encoding="utf-8") as f:
    sea_level_raw = json.load(f)

with open(CONTEXT_STATS_PATH, encoding="utf-8") as f:
    context = json.load(f)

with open(TEMPLATE_PATH, encoding="utf-8") as f:
    template = f.read()

def context_value(key):
    return context[key]["value"]

stats_for_site = [
    {
        "landPct": round(100 - s["land_loss_pct"], 1),
        "pop": s["population_exposed"],
    }
    for s in scenarios_raw["scenarios"]
]
stats_json = json.dumps(stats_for_site, ensure_ascii=False)
sea_level_json = json.dumps(sea_level_raw["series"], ensure_ascii=False)

meta = scenarios_raw["meta"]
baseline_area_km2 = float(meta["baseline_land_area_km2"])
baseline_population = int(round(float(meta["baseline_population"])))
native_worldpop_total = int(round(float(meta["native_clipped_worldpop_total"])))

tuvalu_2223 = int(context_value("tuvalu_resident_population_2022_23"))
funafuti_2223 = int(context_value("funafuti_resident_population_2022_23"))
funafuti_2017 = int(context_value("funafuti_resident_population_2017"))
tuvalu_mean_elev = float(context_value("tuvalu_mean_elevation_msl_m"))

nz_2013 = int(context_value("nz_tuvaluan_ethnic_group_2013"))
nz_2018 = int(context_value("nz_tuvaluan_ethnic_group_2018"))
nz_2023 = int(context_value("nz_tuvaluan_ethnic_group_2023"))
auckland_share = float(context_value("auckland_region_share_tuvaluan_2023_pct"))
pac_2026 = int(context_value("pac_tuvalu_places_2026"))

funafuti_share = funafuti_2223 / tuvalu_2223 * 100
nz_growth = (nz_2023 - nz_2013) / nz_2013 * 100

nz_series = [
    {"year": 2013, "value": nz_2013},
    {"year": 2018, "value": nz_2018},
    {"year": 2023, "value": nz_2023},
]
nz_series_json = json.dumps(nz_series, ensure_ascii=False)

replacements = {
    "__SCENARIO_DATA__": geo_data,
    "__STATS_DATA__": stats_json,
    "__SEA_LEVEL_DATA__": sea_level_json,
    "__NZ_TUVALUAN_SERIES__": nz_series_json,
    "__BASELINE_AREA_KM2__": f"{baseline_area_km2:.3f}",
    "__BASELINE_POPULATION__": f"{baseline_population:,}",
    "__NATIVE_WORLDPOP_TOTAL__": f"{native_worldpop_total:,}",
    "__TUVALU_RESIDENT_2223__": f"{tuvalu_2223:,}",
    "__FUNAFUTI_RESIDENT_2223__": f"{funafuti_2223:,}",
    "__FUNAFUTI_SHARE_2223__": f"{funafuti_share:.1f}",
    "__FUNAFUTI_RESIDENT_2017__": f"{funafuti_2017:,}",
    "__TUVALU_MEAN_ELEVATION__": f"{tuvalu_mean_elev:.2f}",
    "__NZ_TUVALUAN_2013__": f"{nz_2013:,}",
    "__NZ_TUVALUAN_2023__": f"{nz_2023:,}",
    "__NZ_GROWTH_2013_2023__": f"{nz_growth:.0f}",
    "__AUCKLAND_SHARE_2023__": f"{auckland_share:.1f}",
    "__PAC_TUVALU_2026__": f"{pac_2026:,}",
}

missing = [key for key in replacements if key not in template]
if missing:
    raise ValueError(f"Missing template placeholders: {missing}")

print("Scenario rows:", len(stats_for_site))
print("Sea-level observations:", len(sea_level_raw["series"]))
print("Context file verified:", context["meta"]["last_verified"])
print(f"Funafuti share, 2022-23 Census: {funafuti_share:.1f}%")
print(f"NZ Tuvaluan ethnic-group growth, 2013-2023: {nz_growth:.1f}%")

Scenario rows: 5
Sea-level observations: 31
Context file verified: 2026-08-28
Funafuti share, 2022-23 Census: 62.1%
NZ Tuvaluan ethnic-group growth, 2013-2023: 86.2%


## 3. Build HTML

In [3]:
final_html = template
for placeholder, value in replacements.items():
    final_html = final_html.replace(placeholder, value)

PREVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
DEPLOY_PATH.parent.mkdir(parents=True, exist_ok=True)
PREVIEW_PATH.write_text(final_html, encoding="utf-8")
DEPLOY_PATH.write_text(final_html, encoding="utf-8")

print("Built:", PREVIEW_PATH.relative_to(PROJECT_ROOT))
print("Built:", DEPLOY_PATH.relative_to(PROJECT_ROOT))

Built: web\index.html
Built: docs\index.html


## 4. Final checks

In [4]:
expected_scenario_labels = ["Baseline (0.0m)", "+0.5m", "+1.0m", "+1.5m", "+2.0m"]
actual_scenario_labels = [s["label"] for s in scenarios_raw["scenarios"]]

expected_exposed = [0, 0, 201, 201, 532]
actual_exposed = [s["population_exposed"] for s in scenarios_raw["scenarios"]]

expected_land_above = [100.0, 100.0, 84.5, 84.5, 72.3]
actual_land_above = [round(100 - s["land_loss_pct"], 1) for s in scenarios_raw["scenarios"]]

checks = {
    "Five scenario labels": actual_scenario_labels == expected_scenario_labels,
    "Scenario exposed populations": actual_exposed == expected_exposed,
    "Scenario land-above percentages": actual_land_above == expected_land_above,
    "Baseline area is 3.502 km2": round(baseline_area_km2, 3) == 3.502,
    "Baseline model population is 5,905": baseline_population == 5905,
    "Native WorldPop clip rounds to 6,320": native_worldpop_total == 6320,
    "31 annual SPC values": len(sea_level_raw["series"]) == 31,
    "SPC years 1993-2023": [d["year"] for d in sea_level_raw["series"]] == list(range(1993, 2024)),
    "Tuvalu resident population 2022-23": tuvalu_2223 == 10632,
    "Funafuti resident population 2022-23": funafuti_2223 == 6602,
    "Context arithmetic: Funafuti share": abs(funafuti_share - 62.1) < 0.1,
    "NZ Tuvaluan 2013": nz_2013 == 3537,
    "NZ Tuvaluan 2018": nz_2018 == 4653,
    "NZ Tuvaluan 2023": nz_2023 == 6585,
    "Context arithmetic: NZ growth": abs(nz_growth - 86.17) < 0.1,
    "Auckland regional share 2023": abs(auckland_share - 67.1) < 0.01,
    "PAC Tuvalu places 2026": pac_2026 == 75,
    "Tuvalu mean LiDAR elevation": abs(tuvalu_mean_elev - 1.55) < 0.001,
    "Return control present": 'id="wow-return"' in final_html and "Return to Funafuti" in final_html,
    "No forecast-year scenario labels": all(x not in final_html for x in ["~2050", "~2080", "~2100", "Today (2026)"]),
    "No obsolete 4.6 m elevation claim": "4.6m" not in final_html and "4.6 m" not in final_html,
    "No obsolete 10,099 story figure": "10,099" not in final_html,
    "No obsolete model counts": "5,904" not in final_html,
    "No obsolete Auckland local-board claim": "67.2%" not in final_html and "4,422" not in final_html,
    "No unresolved placeholders": all(key not in final_html for key in replacements),
    "web/docs outputs identical": PREVIEW_PATH.read_bytes() == DEPLOY_PATH.read_bytes(),
}

for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {label}")

report = {
    "generated_by": "05_build_html.ipynb",
    "checks": [{"check": label, "status": "PASS" if ok else "FAIL"} for label, ok in checks.items()],
    "summary": {
        "passed": sum(bool(v) for v in checks.values()),
        "total": len(checks),
        "all_passed": all(checks.values()),
    },
    "values": {
        "baseline_area_km2": baseline_area_km2,
        "baseline_population": baseline_population,
        "native_worldpop_total": native_worldpop_total,
        "scenario_population_exposed": actual_exposed,
        "scenario_land_above_pct": actual_land_above,
        "tuvalu_resident_population_2022_23": tuvalu_2223,
        "funafuti_resident_population_2022_23": funafuti_2223,
        "funafuti_share_2022_23_pct": round(funafuti_share, 1),
        "nz_tuvaluan_2013": nz_2013,
        "nz_tuvaluan_2018": nz_2018,
        "nz_tuvaluan_2023": nz_2023,
        "nz_growth_2013_2023_pct": round(nz_growth, 2),
        "auckland_region_share_2023_pct": auckland_share,
        "pac_tuvalu_places_2026": pac_2026,
        "spc_observation_count": len(sea_level_raw["series"]),
    },
}
VALIDATION_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
VALIDATION_REPORT_PATH.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Validation report:", VALIDATION_REPORT_PATH.relative_to(PROJECT_ROOT))

if not all(checks.values()):
    failed = [label for label, ok in checks.items() if not ok]
    raise RuntimeError(f"Build checks failed: {failed}")

print("All build checks passed.")

PASS  Five scenario labels
PASS  Scenario exposed populations
PASS  Scenario land-above percentages
PASS  Baseline area is 3.502 km2
PASS  Baseline model population is 5,905
PASS  Native WorldPop clip rounds to 6,320
PASS  31 annual SPC values
PASS  SPC years 1993-2023
PASS  Tuvalu resident population 2022-23
PASS  Funafuti resident population 2022-23
PASS  Context arithmetic: Funafuti share
PASS  NZ Tuvaluan 2013
PASS  NZ Tuvaluan 2018
PASS  NZ Tuvaluan 2023
PASS  Context arithmetic: NZ growth
PASS  Auckland regional share 2023
PASS  PAC Tuvalu places 2026
PASS  Tuvalu mean LiDAR elevation
PASS  Return control present
PASS  No forecast-year scenario labels
PASS  No obsolete 4.6 m elevation claim
PASS  No obsolete 10,099 story figure
PASS  No obsolete model counts
PASS  No obsolete Auckland local-board claim
PASS  No unresolved placeholders
PASS  web/docs outputs identical
Validation report: output\data_validation_report.json
All build checks passed.


## 5. Preview

After the build, preview `web/index.html` through a local HTTP server. The
`web/index.html` and `docs/index.html` files should be identical.